<a href="https://colab.research.google.com/github/asegura4488/EDCO_MachineLearning/blob/main/Semana6/ProblemaClasificacion_RandomizedSearchCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de *churn* bancario con RandomizedSearchCV

## Objetivo de la clase

Partimos del mismo problema de clasificación binaria del notebook anterior:

- `Exited = 1`: el cliente **abandona** el banco (*churn*).
- `Exited = 0`: el cliente **permanece**.

Ahora queremos responder otra pregunta:

> **¿Cómo elegimos de forma sistemática los hiperparámetros de la red neuronal?**

Usaremos **RandomizedSearchCV** para buscar una buena combinación de arquitectura y parámetros de entrenamiento.

El flujo metodológico será:

1. separar `train / validation / test`;
2. construir un `Pipeline` que contenga **preprocesamiento + red neuronal**;
3. hacer el *tuning* **solo con `train`**, usando validación cruzada estratificada;
4. usar `validation` para seleccionar el umbral de clasificación;
5. usar `test` **una sola vez al final**.

> **Regla central:** `test` no participa ni en el *tuning* de hiperparámetros ni en la selección del *threshold*.

## 1. Instalación e importación de librerías

Para conectar una red de Keras con las herramientas de `scikit-learn` usaremos **SciKeras**.

`KerasClassifier` hace que la red se comporte como un estimador de `scikit-learn`. Gracias a eso podemos colocarla dentro de un `Pipeline` y después pasar ese pipeline a `GridSearchCV` o `RandomizedSearchCV`.

In [ ]:
# Si SciKeras no está instalado, esta celda lo instala.
try:
    import scikeras
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikeras"])

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)

import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout

from scikeras.wrappers import KerasClassifier

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("SciKeras  :", scikeras.__version__)

## 2. Carga de los datos

Se conserva la misma ruta del notebook anterior. Si estás trabajando localmente, modifica `DATA_PATH` para apuntar al archivo `Churn_Modelling.csv`.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_PATH = Path(
        '/content/drive/MyDrive/IntroCienciaDatos/Semana6/Datos/Churn_Modelling.csv'
    )
else:
    DATA_PATH = Path('Datos/Churn_Modelling.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo en: {DATA_PATH}\n'
        'Modifica DATA_PATH para que apunte a Churn_Modelling.csv.'
    )

df = pd.read_csv(DATA_PATH)
print('Dimensiones:', df.shape)
display(df.head())

In [ ]:
# Eliminamos identificadores que no queremos usar como predictores.
cols_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df_model = df.drop(columns=cols_to_drop).copy()

TARGET = 'Exited'
y = df_model[TARGET].astype(int)
X = df_model.drop(columns=TARGET)

summary = pd.DataFrame({
    'n': y.value_counts().sort_index(),
    'proporción': y.value_counts(normalize=True).sort_index(),
})
summary.index = ['Permanece (0)', 'Churn (1)']
display(summary)

## 3. Separación `train / validation / test`

Mantenemos la misma separación del notebook anterior:

- **60 % train**: aquí ocurre el `GridSearchCV`/`RandomizedSearchCV` y su validación cruzada interna;
- **20 % validation**: se reserva para escoger el *threshold* una vez seleccionado el modelo;
- **20 % test**: evaluación final.

Esto puede parecer que crea “dos validaciones”, pero cumplen funciones diferentes:

- los **folds internos de CV** comparan hiperparámetros;
- `X_val` ayuda después a decidir cómo convertir probabilidades en clases 0/1.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=SEED,
    stratify=y_trainval,
)

print('Train:', X_train.shape, '| churn =', round(y_train.mean(), 3))
print('Val  :', X_val.shape,   '| churn =', round(y_val.mean(), 3))
print('Test :', X_test.shape,  '| churn =', round(y_test.mean(), 3))

# 4. Punto conceptual clave: el preprocesamiento debe estar DENTRO del Pipeline

En un *tuning* con validación cruzada no debemos hacer esto:

```python
X_train_p = preprocessor.fit_transform(X_train)
grid.fit(X_train_p, y_train)
```

¿Por qué? Porque `preprocessor.fit_transform(X_train)` habría aprendido medias, desviaciones y categorías usando **todo `X_train` antes de crear los folds**.

La forma correcta es:

```text
Pipeline
├── preprocesamiento
│   ├── StandardScaler
│   └── OneHotEncoder
└── KerasClassifier
```

Entonces, para cada fold, `scikit-learn` ejecuta aproximadamente:

```text
fit(preprocesador, fold_train)
transform(fold_train)
transform(fold_validation)
fit(red, fold_train_transformado)
evaluate(red, fold_validation_transformado)
```

Así, el fold de validación **no participa en el ajuste del preprocesador**.

In [ ]:
# Identificamos tipos de variables usando únicamente X_train.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Compatibilidad con versiones recientes y antiguas de scikit-learn.
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', encoder, categorical_features),
    ],
    remainder='drop',
)

print('Numéricas   :', numeric_features)
print('Categóricas :', categorical_features)

# 5. Convertimos la red Keras en un estimador de scikit-learn

La función `build_model` recibe los hiperparámetros que queremos explorar.

Los cuatro hiperparámetros principales que usaremos son:

| Hiperparámetro | Qué controla | Efecto típico |
|---|---|---|
| `units1` | neuronas de la primera capa oculta | capacidad del modelo |
| `units2` | neuronas de la segunda capa | capacidad/jerarquía de representación |
| `dropout_rate` | fracción de neuronas apagadas durante entrenamiento | regularización |
| `learning_rate` | tamaño de los pasos del optimizador Adam | velocidad/estabilidad del aprendizaje |

> Más neuronas **no significa automáticamente** mejor modelo. Más capacidad puede mejorar el ajuste, pero también aumentar sobreajuste, costo y variabilidad.

In [ ]:
def build_model(
    meta,
    units1=32,
    units2=16,
    dropout_rate=0.20,
    learning_rate=1e-3,
):
    """Construye y compila una red binaria para SciKeras.

    `meta` es suministrado automáticamente por SciKeras.
    Después del preprocesamiento, `meta["n_features_in_"]`
    contiene el número real de columnas que entran a la red.
    """

    n_features = meta['n_features_in_']

    model = Sequential([
        Input(shape=(n_features,)),
        Dense(units1, activation='relu'),
        Dropout(dropout_rate),
        Dense(units2, activation='relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid'),
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(curve='ROC', name='roc_auc'),
        ],
    )

    return model


keras_clf = KerasClassifier(
    model=build_model,
    epochs=60,       # fijo durante la búsqueda para comparar candidatos en igualdad de condiciones
    batch_size=64,
    verbose=0,
    random_state=SEED,
)

pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', keras_clf),
])

pipeline

## ¿Por qué no usamos `EarlyStopping` dentro de esta primera búsqueda?

Para una primera clase de *tuning* conviene mantener el número de épocas fijo. Así, todos los candidatos reciben el mismo presupuesto de entrenamiento.

Es posible combinar validación cruzada y `EarlyStopping`, pero entonces cada fold necesita además una validación interna para decidir cuándo detenerse. Eso introduce una capa adicional de complejidad que puede ocultar la idea principal de `GridSearchCV`/`RandomizedSearchCV`.

Después de comprender este flujo, se puede extender el experimento para incluir `epochs`, callbacks o una validación interna.

# 6. ¿Qué métrica debe decidir cuál modelo es "mejor"?

No usaremos `accuracy` como criterio principal porque la clase positiva (*churn*) es minoritaria.

Calcularemos tres métricas durante la validación cruzada:

- `average_precision`: resume el comportamiento *precision-recall* y será **la métrica usada para seleccionar el mejor modelo**;
- `roc_auc`: capacidad de ordenar positivos por encima de negativos;
- `f1`: equilibrio entre precision y recall, pero depende del umbral interno de clasificación.

La instrucción

```python
refit='average_precision'
```

significa:

> “después de evaluar todos los candidatos, considera ganador al de mayor `average_precision` promedio y vuélvelo a entrenar usando todo `X_train`”.

Elegir `average_precision` tiene una ventaja didáctica adicional: la búsqueda de hiperparámetros no queda atada al umbral 0.5. El *threshold* lo discutiremos después en `validation`.

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED,
)

scoring = {
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc',
    'f1': 'f1',
}

print('Número de folds:', cv.get_n_splits())

# 7. RANDOM SEARCH: la zona de *tuning*

`RandomizedSearchCV` no recorre todas las combinaciones. En cambio, toma **`n_iter` muestras** del espacio de hiperparámetros.

Esto permite explorar espacios mucho más amplios.

Usaremos una mezcla de listas y distribuciones:

- `units1`: valores discretos razonables;
- `units2`: valores discretos razonables;
- `batch_size`: valores discretos;
- `dropout_rate`: una distribución uniforme entre 0 y 0.5;
- `learning_rate`: una distribución **log-uniforme** entre `1e-4` y `5e-3`.

### ¿Por qué log-uniforme para `learning_rate`?

Para tasas de aprendizaje normalmente importan los **órdenes de magnitud**. Por ejemplo,

\[
10^{-4},\; 10^{-3},\; 10^{-2}
\]

son escalas naturalmente diferentes.

Una distribución uniforme ordinaria daría demasiado peso a la región numéricamente grande. `loguniform` explora de forma más equilibrada los órdenes de magnitud.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform

param_distributions = {
    # Arquitectura: conjunto amplio de posibilidades
    'clf__model__units1': [16, 24, 32, 48, 64, 96],
    'clf__model__units2': [8, 12, 16, 24, 32, 48],

    # Dropout continuo entre 0.0 y 0.5
    'clf__model__dropout_rate': uniform(loc=0.0, scale=0.50),

    # Learning rate entre 10^-4 y 5x10^-3 en escala logarítmica
    'clf__model__learning_rate': loguniform(1e-4, 5e-3),

    # Parámetro de fit de SciKeras, no de build_model
    'clf__batch_size': [32, 64, 128],
}

N_ITER = 18
n_folds = cv.get_n_splits()

print('Candidatos aleatorios :', N_ITER)
print('Folds de CV           :', n_folds)
print('Fits aproximados      :', N_ITER * n_folds, '+ 1 refit final')

## ¿Qué hace exactamente `RandomizedSearchCV`?

Conceptualmente:

```text
repetir n_iter veces:
    sortear una configuración θ

    para cada fold k:
        ajustar Pipeline con los folds de entrenamiento
        evaluar sobre el fold dejado fuera

    calcular promedio y dispersión de las métricas

escoger θ* con mayor Average Precision media

refit:
    ajustar Pipeline(θ*) con TODO X_train
```

La diferencia con Grid Search está **antes de entrenar**:

```text
GridSearchCV       -> construye TODAS las combinaciones de una malla finita
RandomizedSearchCV -> toma n_iter muestras del espacio de búsqueda
```

Por eso `n_iter` es el parámetro que controla directamente el presupuesto computacional.

### Los dobles guiones bajos

Por ejemplo:

```python
'clf__model__learning_rate'
```

se interpreta como:

```text
Pipeline
└── clf                 -> paso KerasClassifier
    └── model           -> parámetros enviados a build_model
        └── learning_rate
```

En cambio:

```python
'clf__batch_size'
```

modifica directamente un parámetro de `KerasClassifier.fit`, por eso **no** lleva `model__`.

In [ ]:
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=scoring,
    refit='average_precision',
    cv=cv,
    random_state=SEED,   # hace reproducible el muestreo de hiperparámetros
    n_jobs=1,
    verbose=2,
    return_train_score=False,
    error_score='raise',
)

# AQUÍ OCURRE EL TUNING.
# Se prueban N_ITER configuraciones sorteadas del espacio anterior.
search.fit(X_train, y_train)

# 8. Cómo leer el resultado del *tuning*

Después de `.fit(...)`, los atributos más importantes son:

```python
search.best_params_
search.best_score_
search.best_estimator_
search.cv_results_
```

- `best_params_`: combinación ganadora de hiperparámetros;
- `best_score_`: `average_precision` media de CV para esa combinación;
- `best_estimator_`: pipeline completo ya **reentrenado sobre todo `X_train`** porque usamos `refit='average_precision'`;
- `cv_results_`: tabla completa de todos los candidatos y sus resultados fold a fold.

El valor de `best_score_` **no es una métrica de test**. Es una estimación obtenida dentro del conjunto de entrenamiento mediante validación cruzada.

In [ ]:
print('Mejores hiperparámetros:')
for key, value in search.best_params_.items():
    print(f'  {key}: {value}')

print(f"\nMejor Average Precision media en CV: {search.best_score_:.4f}")

best_model = search.best_estimator_

In [ ]:
# Convertimos los resultados de CV en una tabla ordenada.
results = pd.DataFrame(search.cv_results_)
results = results.sort_values('rank_test_average_precision')

param_cols = [
    c for c in results.columns
    if c.startswith('param_')
]

show_cols = (
    ['rank_test_average_precision']
    + param_cols
    + [
        'mean_test_average_precision',
        'std_test_average_precision',
        'mean_test_roc_auc',
        'mean_test_f1',
        'mean_fit_time',
    ]
)

display(results[show_cols].head(10))

### ¿Qué significa `mean_test_average_precision` aquí?

El nombre `test` dentro de `cv_results_` puede confundir.

**No se refiere a nuestro `X_test` final.**

En la terminología de `GridSearchCV`/`RandomizedSearchCV`, `test` significa simplemente **el fold que se dejó fuera para validar en cada iteración de CV**.

Por ejemplo, con 3 folds:

```text
Iteración 1: train = folds 2+3   | validación interna = fold 1
Iteración 2: train = folds 1+3   | validación interna = fold 2
Iteración 3: train = folds 1+2   | validación interna = fold 3
```

Luego:

\[
\overline{AP}(	heta)=rac{AP_1(	heta)+AP_2(	heta)+AP_3(	heta)}{3}.
\]

El candidato con mayor promedio es el ganador.

In [ ]:
# Visualización de los mejores candidatos.
top = results.head(10).copy().iloc[::-1]

plt.figure(figsize=(8, 5))
plt.barh(
    range(len(top)),
    top['mean_test_average_precision'],
    xerr=top['std_test_average_precision'],
)
plt.yticks(range(len(top)), [f'Candidato {i}' for i in top.index])
plt.xlabel('Average Precision media en CV')
plt.title('Mejores candidatos del tuning')
plt.show()

# 9. Selección del threshold usando VALIDATION

Hasta este punto seleccionamos **hiperparámetros**, pero la red aún produce probabilidades/scores.

Ahora usamos `X_val` para decidir el umbral que maximiza F1. Esta decisión **no se toma dentro de test**.

In [ ]:
def positive_class_probability(estimator, X_data):
    """Devuelve P(Y=1) de forma robusta para un clasificador binario."""
    proba = np.asarray(estimator.predict_proba(X_data))

    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]
    return proba.ravel()


def metrics_at_threshold(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return pd.Series({
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    })


y_val_prob = positive_class_probability(best_model, X_val)

print(f'Validation ROC-AUC: {roc_auc_score(y_val, y_val_prob):.4f}')
print(f'Validation AP     : {average_precision_score(y_val, y_val_prob):.4f}')

In [ ]:
prec_v, rec_v, thresholds_v = precision_recall_curve(y_val, y_val_prob)

prec_for_t = prec_v[:-1]
rec_for_t = rec_v[:-1]
f1_for_t = 2 * prec_for_t * rec_for_t / (prec_for_t + rec_for_t + 1e-12)

best_idx = np.argmax(f1_for_t)
best_threshold = float(thresholds_v[best_idx])
best_val_f1 = float(f1_for_t[best_idx])

print(f'Mejor threshold en validation: {best_threshold:.4f}')
print(f'F1 en ese punto             : {best_val_f1:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(thresholds_v, f1_for_t)
plt.scatter([best_threshold], [best_val_f1], zorder=3)
plt.xlabel('Threshold')
plt.ylabel('F1')
plt.title('Selección del threshold usando validation')
plt.show()

comparison_val = pd.DataFrame([
    metrics_at_threshold(y_val, y_val_prob, 0.50),
    metrics_at_threshold(y_val, y_val_prob, best_threshold),
])
comparison_val.index = ['threshold = 0.50', 'threshold optimizado en validation']
display(comparison_val)

# 10. Evaluación final sobre TEST

Ahora sí usamos el conjunto de prueba.

A partir de aquí no deberíamos seguir modificando hiperparámetros ni threshold basándonos en estos resultados. Si lo hacemos repetidamente, `test` deja de representar datos realmente no vistos.

In [ ]:
y_test_prob = positive_class_probability(best_model, X_test)
y_test_pred = (y_test_prob >= best_threshold).astype(int)

print(f'Test ROC-AUC: {roc_auc_score(y_test, y_test_prob):.4f}')
print(f'Test AP     : {average_precision_score(y_test, y_test_prob):.4f}')
print()

print(classification_report(
    y_test,
    y_test_pred,
    target_names=['No churn', 'Churn'],
    digits=3,
    zero_division=0,
))

comparison_test = pd.DataFrame([
    metrics_at_threshold(y_test, y_test_prob, 0.50),
    metrics_at_threshold(y_test, y_test_prob, best_threshold),
])
comparison_test.index = ['threshold = 0.50', 'threshold elegido en validation']
display(comparison_test)

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cbar=False,
    xticklabels=['No churn', 'Churn'],
    yticklabels=['No churn', 'Churn'],
)
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title(f'Matriz de confusión - test (t = {best_threshold:.3f})')
plt.show()

# 11. Ideas para discutir en clase

1. ¿Por qué el preprocesador debe estar dentro del `Pipeline`?
2. ¿Qué diferencia existe entre los folds internos de CV, `X_val` y `X_test`?
3. ¿Por qué escogimos `average_precision` como criterio de `refit`?
4. ¿Qué cambiaría si usáramos `refit='f1'`?
5. ¿Por qué `f1` sí depende de un threshold mientras ROC-AUC/AP trabajan con scores continuos?
6. ¿Qué significa que dos configuraciones tengan medias parecidas pero desviaciones estándar diferentes?
7. ¿Por qué `best_score_` no debe reportarse como “desempeño final del modelo”?
8. ¿Cuándo sería razonable incluir `batch_size`, `epochs` o `class_weight` dentro del espacio de búsqueda?

## Inspeccionar qué configuraciones fueron realmente sorteadas

Esta tabla es especialmente útil en `RandomizedSearchCV`, porque los candidatos no estaban escritos explícitamente uno por uno antes del entrenamiento.

In [ ]:
random_draws = results[param_cols].copy()
display(random_draws.head(18))

## Resumen conceptual: RandomizedSearchCV

`RandomizedSearchCV` suele ser preferible cuando:

- hay muchos hiperparámetros;
- algunos son continuos (`learning_rate`, `dropout`, regularización, etc.);
- entrenar cada candidato es costoso;
- queremos controlar el presupuesto con `n_iter`.

Una idea importante es que **Random Search no significa una búsqueda de baja calidad**. En espacios de alta dimensión, puede usar el presupuesto de forma mucho más eficiente que una malla rígida, porque no obliga a probar todas las combinaciones posibles entre valores que quizá tengan poca influencia.

Una estrategia práctica común es:

```text
1. RandomizedSearchCV amplio
2. identificar una región prometedora
3. GridSearchCV pequeño alrededor de esa región
```

Así combinamos exploración amplia con refinamiento local.